In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from statsmodels.tsa.arima.model import ARIMA
from pathlib import Path

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
fontsize = 16
BLUE  = "#2D6B8F"
CORAL = "#CA6F6A"

T_TOTAL = 1500
PERIOD  = 200
W, H    = 100, 20
STEP    = 5

rng    = np.random.default_rng(42)
t      = np.arange(0, T_TOTAL, STEP)
series = np.where((t % PERIOD) < PERIOD // 2, 150.0, 50.0).astype(float)
series += rng.normal(0, 2.5, T_TOTAL)[::STEP]

W_s = W // STEP
H_s = H // STEP
starts_idx = np.arange(W_s, len(series) - H_s)
starts     = t[starts_idx]

def arima_forecast(ctx_norm, h):
    try:
        return ARIMA(ctx_norm, order=(2, 0, 0)).fit().forecast(steps=h)
    except Exception:
        return np.full(h, ctx_norm.mean())

mae_lag, mae_caus = [], []
for i, idx in enumerate(starts_idx):
    ctx = series[idx - W_s : idx]
    fut = series[idx : idx + H_s]

    mu_lag, sig_lag = ctx.mean(), ctx.std().clip(1e-6)
    fcast_lag       = arima_forecast((ctx - mu_lag) / sig_lag, H_s)
    fut_norm_lag    = (fut - mu_lag) / sig_lag
    mae_lag.append(np.abs(fcast_lag - fut_norm_lag).mean())

    mu_caus, sig_caus = series[:idx].mean(), series[:idx].std().clip(1e-6)
    fcast_caus        = arima_forecast((ctx - mu_caus) / sig_caus, H_s)
    fut_norm_caus     = (fut - mu_caus) / sig_caus
    mae_caus.append(np.abs(fcast_caus - fut_norm_caus).mean())

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(starts_idx)}")

mae_lag  = np.array(mae_lag)
mae_caus = np.array(mae_caus)
y_max    = mae_lag.max() * 1.1

DOT_EVERY = 2
dot_idx   = np.arange(0, len(starts), DOT_EVERY)

def make_fig(mae, path=None):
    fig, ax1 = plt.subplots(figsize=(10, 3))
    fig.patch.set_facecolor("white")

    ax1.plot(t, series, color=BLUE, lw=1.2, alpha=0.7, zorder=2)
    ax1.set_ylabel("y (Original Space)", color=BLUE, fontsize=fontsize)
    ax1.tick_params(axis='y', labelcolor=BLUE, labelsize=fontsize)
    ax1.tick_params(axis='x', labelcolor='black', labelsize=fontsize)
    ax1.set_xlabel("t (Forecast Creation Date)", fontsize=fontsize, color='black')
    ax1.set_xlim(0, T_TOTAL)
    for sp in ax1.spines.values():
        sp.set_edgecolor("#e0e0e0")

    ax2 = ax1.twinx()
    ax2.set_ylim(0, y_max)
    ax2.set_xlim(0, T_TOTAL)
    ax2.plot(starts, mae, color=CORAL, lw=1.2, alpha=0.85, zorder=3)
    ax2.plot(starts[dot_idx], mae[dot_idx], 'o', color=CORAL, ms=3.5, zorder=4)
    ax2.set_ylabel("MAE (Norm. Space)", color=CORAL, fontsize=fontsize)
    ax2.tick_params(axis='y', labelcolor=CORAL, labelsize=fontsize)
    ax2.spines['right'].set_edgecolor('CORAL')
    ax2.spines['right'].set_alpha(0.5)

    plt.grid(False)
    plt.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')

make_fig(mae_lag,  path="./ws_lag_norm_synthetic.pdf")
make_fig(mae_caus, path="./ws_causal_norm_synthetic.pdf")

In [6]:
# from datasetsforecast.favorita import FavoritaData

# group = 'Favorita200' # 'Favorita500', 'FavoritaComplete'
# directory = './data/favorita'
# Y_df, S_df, tags = FavoritaData.load(directory=directory, group=group)

df = pd.read_csv('/home/wpotosna/forgets/figures/data/favorita/train.csv')
item = df[(df['item_nbr'] == 105574)&(df['store_nbr']==5)]

/tmp/ipykernel_2586788/2757849500.py:7: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/wpotosna/forgets/figures/data/favorita/train.csv')


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
fontsize = 16
BLUE  = "#2D6B8F"
CORAL = "#CA6F6A"

raw_series = item['unit_sales'].values.astype(float)
T_TOTAL    = len(raw_series)
t          = np.arange(T_TOTAL)
series     = raw_series

W, H = 100, 20
W_s  = W
H_s  = H
starts_idx = np.arange(W_s, len(series) - H_s)
starts     = t[starts_idx]

def arima_forecast(ctx_norm, h):
    try:
        return ARIMA(ctx_norm, order=(2, 0, 0)).fit().forecast(steps=h)
    except Exception:
        return np.full(h, ctx_norm.mean())

mae_lag, mae_caus = [], []
for i, idx in enumerate(starts_idx):
    ctx = series[idx - W_s : idx]
    fut = series[idx : idx + H_s]

    mu_lag, sig_lag = ctx.mean(), ctx.std().clip(1e-6)
    fcast_lag       = arima_forecast((ctx - mu_lag) / sig_lag, H_s)
    fut_norm_lag    = (fut - mu_lag) / sig_lag
    mae_lag.append(np.abs(fcast_lag - fut_norm_lag).mean())

    mu_caus, sig_caus = series[:idx].mean(), series[:idx].std().clip(1e-6)
    fcast_caus        = arima_forecast((ctx - mu_caus) / sig_caus, H_s)
    fut_norm_caus     = (fut - mu_caus) / sig_caus
    mae_caus.append(np.abs(fcast_caus - fut_norm_caus).mean())

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(starts_idx)}")

mae_lag  = np.array(mae_lag)
mae_caus = np.array(mae_caus)
y_max    = mae_lag.max() * 1.1

DOT_EVERY = 2
dot_idx   = np.arange(0, len(starts), DOT_EVERY)

def make_fig(mae, path=None):
    fig, ax1 = plt.subplots(figsize=(10, 3))
    fig.patch.set_facecolor("white")

    ax1.plot(t, series, color=BLUE, lw=1.2, alpha=0.7, zorder=2)
    ax1.set_ylabel("y (Original Space)", color=BLUE, fontsize=fontsize)
    ax1.tick_params(axis='y', labelcolor=BLUE, labelsize=fontsize)
    ax1.tick_params(axis='x', labelcolor='black', labelsize=fontsize)
    ax1.set_xlabel("t (Forecast Creation Date)", fontsize=fontsize, color='black')
    ax1.set_xlim(0, T_TOTAL)
    for sp in ax1.spines.values():
        sp.set_edgecolor("#e0e0e0")

    ax2 = ax1.twinx()
    ax2.set_ylim(0, y_max)
    ax2.set_xlim(0, T_TOTAL)
    ax2.plot(starts, mae, color=CORAL, lw=1.2, alpha=0.85, zorder=3)
    ax2.plot(starts[dot_idx], mae[dot_idx], 'o', color=CORAL, ms=3.5, zorder=4)
    ax2.set_ylabel("MAE (Norm. Space)", color=CORAL, fontsize=fontsize)
    ax2.tick_params(axis='y', labelcolor=CORAL, labelsize=fontsize)
    ax2.spines['right'].set_edgecolor('CORAL')
    ax2.spines['right'].set_alpha(0.5)

    plt.grid(False)
    plt.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')

make_fig(mae_lag,  path="./ws_lag_norm_favorita.pdf")
make_fig(mae_caus, path="./ws_causal_norm_favorita.pdf")